# Практична робота № 3


## Крок 1. Підготовка середовища та завантаження даних


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/index_1.csv')
print('Розмірність:', df.shape)
df.head()

Розмірність: (3636, 6)


,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


## Крок 2. Виявлення та обробка пропущених значень

In [2]:
# Синтетичні дані
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

,name,age,score
0,Anna,20.0,85.5
1,None,21.0,90.0
2,Bohdan,NaN,78.3
3,Olena,22.0,NaN
4,None,24.0,92.1


In [3]:
print(df_messy.isnull().sum())  # кількість пропусків у кожній колонці

name     2
age      1
score    1
dtype: int64


In [4]:
# Видалення рядків з пропусками у колонці 'name'
df_dropped = df_messy.dropna(subset=['name'])  # видаляє рядки, де в name пропуск
print(df_dropped)

# Заповнення пропусків у колонці 'age' середнім значенням
mean_age = df_messy['age'].mean()  # середнє, ігноруючи NaN
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)  # заміна пропусків середнім
print(df_filled)

     name   age  score
0    Anna  20.0   85.5
2  Bohdan   NaN   78.3
3   Olena  22.0    NaN
     name    age  score
0    Anna  20.00   85.5
1    None  21.00   90.0
2  Bohdan  21.75   78.3
3   Olena  22.00    NaN
4    None  24.00   92.1


## Крок 3. Виявлення та видалення дублікатів

In [5]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

,id,value
0,1,A
1,2,B
2,3,C
3,2,B
4,3,C


In [6]:
print(df_dup.duplicated())
print("Кількість дублікатів:", df_dup.duplicated().sum())

0    False
1    False
2    False
3     True
4     True
dtype: bool
Кількість дублікатів: 2


In [7]:
df_unique = df_dup.drop_duplicates()  # залишає перше входження кожного рядка
print(df_unique)

   id value
0   1     A
1   2     B
2   3     C


## Крок 4. Виявлення та обробка викидів (метод IQR)

In [8]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

Q1 = 27.92, Q3 = 35.76, IQR = 7.839999999999996
Нижня межа: 16.160000000000007, Верхня межа: 47.519999999999996


In [9]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

Знайдено викидів: 0


## Крок 5. Перетворення типів даних

In [10]:
# Перевірка типів
print(df.dtypes, "\n")

# Перетворення datetime
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

# Перевірка типів після перетворення
print(df.dtypes)

date            object
datetime        object
cash_type       object
card            object
money          float64
coffee_name     object
dtype: object 

date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object


## Крок 6. Обробка рядкових даних

In [11]:
df_text = pd.DataFrame({
    'city': ['  Kyiv  ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89', '(057) 456-78-90', '(044) 123-45-67']
})
df_text

,city,phone
0,Kyiv,(044) 123-45-67
1,LVIV,(032) 234-56-78
2,Odesa,(048) 345-67-89
3,KHARKIV,(057) 456-78-90
4,kyiv,(044) 123-45-67


In [12]:
df_text['city'] = df_text['city'].str.strip().str.lower()
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)
print(df_text)

      city        phone
0     kyiv  044 1234567
1     lviv  032 2345678
2    odesa  048 3456789
3  kharkiv  057 4567890
4     kyiv  044 1234567


## Крок 7. Створення нових ознак (feature engineering)

In [13]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

                 datetime   date_only  hour  day_of_week  month
0 2024-03-01 10:15:50.520  2024-03-01    10            4      3
1 2024-03-01 12:19:22.539  2024-03-01    12            4      3
2 2024-03-01 12:20:18.089  2024-03-01    12            4      3
3 2024-03-01 13:46:33.006  2024-03-01    13            4      3
4 2024-03-01 13:48:14.626  2024-03-01    13            4      3


## Крок 8. One-hot encoding категоріальних змінних

In [14]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

         date                datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 10:15:50.520      card  ANON-0000-0000-0001   38.7   
1  2024-03-01 2024-03-01 12:19:22.539      card  ANON-0000-0000-0002   38.7   
2  2024-03-01 2024-03-01 12:20:18.089      card  ANON-0000-0000-0002   38.7   
3  2024-03-01 2024-03-01 13:46:33.006      card  ANON-0000-0000-0003   28.9   
4  2024-03-01 2024-03-01 13:48:14.626      card  ANON-0000-0000-0004   38.7   

    date_only  hour  day_of_week  month  coffee_name_Americano  \
0  2024-03-01    10            4      3                  False   
1  2024-03-01    12            4      3                  False   
2  2024-03-01    12            4      3                  False   
3  2024-03-01    13            4      3                   True   
4  2024-03-01    13            4      3                  False   

   coffee_name_Americano with Milk  coffee_name_Cappuccino  coffee_name_Cocoa  \
0                            False             

## Крок 9. Нормалізація та стандартизація

In [15]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()

In [16]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

   money  money_norm  money_std
0   38.7    0.940585   1.413261
1   38.7    0.940585   1.413261
2   38.7    0.940585   1.413261
3   28.9    0.492687  -0.578639
4   38.7    0.940585   1.413261
Мінімум money_norm: 0.0
Максимум money_norm: 1.0
Середнє money_std: 1.1725127652806934e-17
Стандартне відхилення money_std: 1.0000000000000462


## Крок 10. Збереження очищених даних

In [17]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name', 'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv('/content/cleaned_sales.csv', index=False)
final_df.head()

,datetime,cash_type,money,coffee_name,hour,day_of_week,month,money_norm,money_std
0,2024-03-01 10:15:50.520,card,38.7,Latte,10,4,3,0.940585,1.413261
1,2024-03-01 12:19:22.539,card,38.7,Hot Chocolate,12,4,3,0.940585,1.413261
2,2024-03-01 12:20:18.089,card,38.7,Hot Chocolate,12,4,3,0.940585,1.413261
3,2024-03-01 13:46:33.006,card,28.9,Americano,13,4,3,0.492687,-0.578639
4,2024-03-01 13:48:14.626,card,38.7,Latte,13,4,3,0.940585,1.413261


## Індивідуальне завдання (варіант 7)
Створіть нову числову ознаку `is_card`, яка дорівнює 1, якщо `cash_type == 'card'`, і 0 інакше. Використайте `np.where()`. Виведіть перші десять рядків із колонками `cash_type` та `is_card`.

In [18]:
df['is_card'] = np.where(df['cash_type'] == 'card', 1, 0)
df[['cash_type', 'is_card']].head(10)

,cash_type,is_card
0,card,1
1,card,1
2,card,1
3,card,1
4,card,1
5,card,1
6,card,1
7,card,1
8,card,1
9,card,1


## ★ Додаткове завдання: функція `clean_dataset(df)`
Функція повертає очищену копію: видаляє повні дублікати, перетворює колонки з датою на `datetime64`, заповнює числові пропуски медіаною, категоріальні — модою, і видаляє рядки з викидами за IQR для всіх числових колонок.

In [19]:
def clean_dataset(df, date_cols=None):
    '''Повертає очищену копію DataFrame.'''
    out = df.copy()

    # 1. Повні дублікати
    out = out.drop_duplicates()

    # 2. Колонки з датою -> datetime64 (за замовчуванням ті, що містять 'date' або 'time' у назві)
    if date_cols is None:
        date_cols = [c for c in out.columns if 'date' in str(c).lower() or 'time' in str(c).lower()]
    for col in date_cols:
        out[col] = pd.to_datetime(out[col], errors='coerce')

    # 3. Пропуски: числові -> медіана, категоріальні -> мода
    num_cols = out.select_dtypes(include='number').columns
    cat_cols = [c for c in out.columns if c not in num_cols and c not in date_cols]
    for col in num_cols:
        out[col] = out[col].fillna(out[col].median())
    for col in cat_cols:
        mode = out[col].mode()
        if not mode.empty:
            out[col] = out[col].fillna(mode.iloc[0])

    # 4. Викиди за IQR для всіх числових колонок
    for col in num_cols:
        Q1, Q3 = out[col].quantile(0.25), out[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        out = out[(out[col] >= lower) & (out[col] <= upper)]

    return out.reset_index(drop=True)

Тестуємо на штучно створеному «брудному» датасеті: є пропуски, дублікати, викиди та дати у вигляді тексту.

In [20]:
rng = np.random.default_rng(1)
n = 30
dirty = pd.DataFrame({
    'date': pd.date_range('2024-03-01', periods=n, freq='D').strftime('%Y-%m-%d'),
    'product': rng.choice(['Latte', 'Americano', 'Cappuccino'], n),
    'city': rng.choice(['Kyiv', 'Lviv', 'Odesa'], n),
    'age': rng.integers(18, 60, n).astype(float),
    'salary': rng.normal(20000, 3000, n).round(2),
})
dirty.loc[[3, 8, 15], 'age'] = np.nan          # пропуски (числові)
dirty.loc[[5, 11], 'city'] = np.nan            # пропуски (категоріальні)
dirty.loc[[2, 20], 'salary'] = [500000, -90000]  # викиди
dirty = pd.concat([dirty, dirty.iloc[[0, 1, 4]]], ignore_index=True)  # 3 дублікати

print('До очищення:', dirty.shape)
print(dirty.isnull().sum(), '\n')

cleaned = clean_dataset(dirty)

print('Після очищення:', cleaned.shape)
print(cleaned.isnull().sum(), '\n')
print(cleaned.dtypes)
cleaned.head()

До очищення: (33, 5)
date       0
product    0
city       2
age        3
salary     0
dtype: int64 

Після очищення: (28, 5)
date       0
product    0
city       0
age        0
salary     0
dtype: int64 

date       datetime64[ns]
product            object
city               object
age               float64
salary            float64
dtype: object


,date,product,city,age,salary
0,2024-03-01,Americano,Kyiv,35.0,19705.19
1,2024-03-02,Americano,Lviv,39.0,20286.45
2,2024-03-04,Cappuccino,Kyiv,39.0,18481.13
3,2024-03-05,Latte,Lviv,35.0,21781.24
4,2024-03-06,Latte,Kyiv,44.0,22673.50


In [21]:
cleaned.to_csv('/content/cleaned_dirty.csv', index=False)
print('Файл збережено!')

Файл збережено!
